# 05 - Risk assessment, future prediction and decision support
Uses the outputs of notebook 04 (two land-cover maps and two indicator stacks on the same grid).

| Part | Method |
|---|---|
| Flood risk | Monsoon-only water (Sentinel-1 radar) nearby + low-lying + flat ground |
| Drought risk | 1 - Vegetation Health Index (dry-season NDVI and Landsat LST), vegetated land only |
| Deforestation risk | Remaining trees near recent tree loss and new built-up land |
| Future land cover | CA-Markov: 2019->2025 transition rates, placed by neighbourhood (predicts 2031) |
| Decision support | 2 km grid cells ranked by priority, with main issue and suggested action |

These are screening indices, not calibrated hazard models - state this in the report.

In [ ]:
# --- Setup (run once per Colab session) ---
!git clone -q https://github.com/veeresh0804/PrithviBodh.git 2>/dev/null || (cd PrithviBodh && git pull -q)
%cd /content/PrithviBodh
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PrithviBodh'      # Earth Engine exports land here
import os; os.makedirs(DRIVE, exist_ok=True)

import sys; sys.path.insert(0, '.')
from prithvibodh.config import load_config
cfg = load_config()
cfg['gee_project'] = 'YOUR_EE_PROJECT_ID'         # <-- your free noncommercial project
from prithvibodh import gee_data as G
G.init(cfg)

In [ ]:
import numpy as np, pandas as pd
from prithvibodh import rasters, risk, prediction, decision, change
from prithvibodh.gee_data import INDICATOR_BANDS, INDICATOR_SCALE
y0, y1 = cfg['years']
m0, prof = rasters.read(f'{DRIVE}/landcover_{y0}.tif')
m1, _ = rasters.read(f'{DRIVE}/landcover_{y1}.tif')
ind = rasters.read_bands(f'{DRIVE}/indicators_{y1}.tif', INDICATOR_BANDS, INDICATOR_BANDS, INDICATOR_SCALE)
assert m1.shape == ind['ndvi_dry'].shape, 'maps and indicators must be on the same grid (see rasters.match_grid)'
valid = m1 != 255
cuts = cfg['risk']['levels']

In [ ]:
flood = risk.flood_risk(ind['water_dry'], ind['water_wet'], ind['elevation'], ind['slope'], valid, cfg)
drought = risk.drought_risk(ind['ndvi_dry'], ind['lst_dry'], m1, cfg['classes'], valid)
defor = risk.deforestation_risk(m0, m1, cfg['classes'], cfg)
for name, sc in [('flood', flood), ('drought', drought), ('deforestation', defor)]:
    rasters.write(f'{DRIVE}/risk_{name}.tif', rasters.to_uint8_score(sc), prof)
pd.DataFrame({n: risk.summary(sc, cuts) for n, sc in
              [('flood', flood), ('drought', drought), ('deforestation', defor)]}).T.round(1)

In [ ]:
# Future land cover (CA-Markov)
P = prediction.transition_probabilities(m0, m1, len(cfg['classes']))
steps = cfg['prediction']['steps']
future_year = y1 + steps * (y1 - y0)
proj = prediction.project_areas(m1, P, steps, cfg['classes'])
proj.to_csv(f'{DRIVE}/projection_{future_year}.csv')
future = prediction.ca_markov(m1, P, steps, cfg['prediction']['window_px'])
rasters.write(f'{DRIVE}/landcover_{future_year}_predicted.tif', future, prof)
print('Transition probabilities per interval:'); display(pd.DataFrame(P, index=cfg['classes'], columns=cfg['classes']).round(3))
proj

In [ ]:
# Decision support: ranked 2 km cells
t = prof['transform']
table = decision.priority_table(m0, m1, cfg['classes'], flood, drought, defor,
                                change.degradation_mask(m0, m1, cfg['classes']),
                                (t.a, t.b, t.c, t.d, t.e, t.f), cfg)
if prof['crs'] and prof['crs'].is_projected:          # Option B maps: convert cell centres to lon/lat
    from rasterio.warp import transform as tr
    table['x'], table['y'] = tr(prof['crs'], 'EPSG:4326', table['x'].tolist(), table['y'].tolist())
table = table.rename(columns={'x': 'lon', 'y': 'lat'})
table.to_csv(f'{DRIVE}/priority_cells.csv', index=False)
table.head(cfg['decision']['top_n'])[['lon', 'lat', 'priority', 'main_issue', 'suggested_action']]

In [ ]:
!cp {DRIVE}/risk_*.tif {DRIVE}/landcover_*_predicted.tif {DRIVE}/projection_*.csv {DRIVE}/priority_cells.csv outputs/ 2>/dev/null; ls -lh outputs